# V-SHIELD Anti-Spoofing Model Evaluation & Analysis

This notebook provides interactive evaluation, ROC curve plotting, and threshold tuning for V-SHIELD checkpoints.

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import soundfile as sf

# Ensure project root in path
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
if os.path.join(repo_root, "ml", "src") not in sys.path:
    sys.path.insert(0, os.path.join(repo_root, "ml", "src"))

from model import LightweightAntiSpoofCNN
from training.metrics import compute_eer, compute_classification_metrics, compute_far_frr
from training.audio import preprocess_audio, load_audio

In [ ]:
# Load Model Checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = "../models/vshield_antispoof_v1/best_model.pt"

model = LightweightAntiSpoofCNN().to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
model.eval()
print(f"Loaded model from {ckpt_path} onto {device}")

In [ ]:
# Single Audio File Inference Test
def test_single_audio(file_path):
    waveform, sr = load_audio(file_path, target_sr=16000)
    processed = preprocess_audio(waveform, target_length=64000, mode="deterministic").unsqueeze(0).to(device)
    with torch.no_grad():
        logit = model(processed)
        prob = torch.sigmoid(logit).item()
    print(f"File: {file_path} | Spoof Probability: {prob:.4f} ({'SPOOF' if prob >= 0.5 else 'BONAFIDE'})")
    return prob